# Course notebook — Machine Learning Systems

Learn how models are trained, validated, diagnosed, and integrated into reliable software systems.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The repairs that arrive after moving day

Daniel coordinates maintenance for a housing cooperative. Residents sometimes wait several days for a repair because the mobile form sends sparse descriptions into the same queue as rich phone reports. The board funds one pilot. Daniel can promise faster triage only if mobile requests and uncommon repair types see the same benefit.

The workplace scenario is fictional.


## Problem framing and supervised models


### Lecture 01 · Problems, Data, and Baselines

Frame an ML task before choosing an algorithm. Define the unit of prediction, target, decision, population, time boundary, data availability, and a simple baseline.

#### Define urgency before opening a notebook

The historical priority field reflects which jobs a supervisor eventually expedited. It also reflects who called repeatedly. Training directly on that field could teach the system to give quieter residents slower service.

Choose an operational label such as whether verified conditions demand a same-day visit. Establish a rules baseline using information available at intake. Inspect a small sample with the maintenance team before deciding whether older labels can be reused.

#### Read the mechanism

**problem framing**

Problem framing defines the unit of prediction, target label, decision, population, and time boundary before choosing algorithms. Most ML failures are mis-specified problems, not wrong models.

Predicting 'will this ticket reopen within 7 days' differs from 'summarize this ticket'—only the first is a measurable ML task.

**features and labels**

Features are inputs; labels are supervised targets—both must be available at the decision time you actually deploy. Leaking future information creates impressive offline metrics and production disasters.

Using 'time to resolution' as a feature to predict escalation leaks the outcome into the input.

**sampling**

Sampling draws next tokens from the predicted distribution rather than always taking the argmax. It enables diverse outputs but introduces nondeterminism unless seeded.

Creative writing uses sampling; factual extraction often uses greedy or low-temperature decoding.

**data leakage**

Data leakage lets information from the target or future timesteps into features or labels during training. It inflates offline metrics while production performance collapses.

Including the support agent's resolution note written after closure as a feature perfectly predicts reopen—uselessly.

**baselines**

Baselines are simple reference methods—majority class, linear model, keyword rules—that quantify what complexity must beat. Without them, teams cannot justify neural networks or LLMs.

A TF–IDF logistic regression baseline on ticket routing sets the bar before trying embeddings.

#### Worked example

**Situation:** A lender needs a prediction service whose errors can be explained across customer groups. Product asks for "approve/deny" but data only has past committee decisions.

**Baseline:** Predict majority class (approve) for every application—high accuracy, useless for risk.

**Application:** Frame unit of prediction (application at submission time), label (committee decision within 30 days), forbid future payment behavior as features, split by application date and customer entity, and ship a frequency baseline before any complex model.

**Test cases:** (1) Normal: complete application with stable income fields. (2) Boundary: application submitted at midnight UTC boundary. (3) Adversarial: duplicate applications with synchronized IDs leaking target via entity overlap in train and test.

**Measurement:** AUC vs baseline, slice metrics by region, and leakage audit checklist pass/fail.

**Design question:** Which feature would you ban first after a leakage review, and how would slice metrics expose it?

**Predict before running.** Most model failures begin as problem or data-definition failures. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 2.1: Problems, Data, and Baselines"""

apps = [
    {"id": 1, "income": 80000, "decision": 1},
    {"id": 2, "income": 40000, "decision": 0},
    {"id": 3, "income": 120000, "decision": 1},
]
baseline_rate = sum(a["decision"] for a in apps) / len(apps)
pred = 1 if baseline_rate >= 0.5 else 0
acc = sum(pred == a["decision"] for a in apps) / len(apps)
print({"majority_pred": pred, "accuracy": round(acc, 3)})


#### Interpret and apply

Create a dataset split that respects time and entity boundaries.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What problem remains if you optimize accuracy without defining the prediction unit?
2. How would entity leakage differ from temporal leakage in metrics?
3. What baseline must every lending model beat before release?

<details><summary>Check your reasoning</summary>

Q1: Wrong granularity makes metrics non-actionable and hides cohort drift. Q2: Entity leakage inflates all slices; temporal leakage shows val superiority on future-dated features. Q3: Majority-class baseline with same split protocol.

</details>


### Lecture 02 · Supervised Learning

Understand regression and classification as function approximation under a chosen loss. Connect linear models, trees, and neural networks through their assumptions.

#### What gets more expensive when a threshold moves?

The model sees 100 urgent and 900 ordinary requests. Moving the priority threshold catches 8 more urgent cases and adds 35 ordinary cases to the priority queue.

The policy trade-off is measurable. Estimate effects on urgent misses and technician capacity separately, then ask who can absorb the additional reviews. A change that increases recall can still fail if it creates a queue long enough to delay all urgent work.

#### Read the mechanism

**regression**

Regression predicts continuous targets—latency, revenue, temperature—by minimizing loss over numeric outputs. Choice of loss (MSE, Huber) reflects outlier sensitivity in operations.

Forecasting queue wait time uses regression; thresholds on predicted minutes trigger staffing alerts.

**classification**

Classification assigns inputs to discrete categories via scores converted to labels. Thresholds, class imbalance, and cost asymmetry matter as much as raw accuracy.

Binary fraud classification at 0.5 default threshold wastes money when false positives cost $2 and false negatives cost $200.

**loss functions**

Loss functions score how wrong predictions are and drive optimization—cross-entropy for classes, MSE for regression, custom losses for ranking. The loss encodes what the system is punished for.

Using focal loss down-weights easy negatives so a rare-defect detector trains on hard examples.

**regularization**

Regularization penalizes complexity—L2 weight decay, dropout, early stopping—to improve generalization. It trades training fit for deployment stability.

Dropout on a small tabular network prevents memorizing 500 rows of customer data.

**optimization**

Optimization finds parameters that minimize loss—SGD, Adam, learning-rate schedules, and batch size interact with convergence speed and final quality.

A too-high learning rate oscillates; too-low wastes GPU hours on a plateau.

#### Worked example

**Situation:** The lender must predict default risk from twelve numeric features with interpretability requirements for compliance.

**Baseline:** Linear logistic regression with L2 regularization—coefficients readable by auditors.

**Application:** Implement linear and small tree models, compare loss curves, inspect coefficient signs vs domain expectations, and choose the simplest meeting recall@deny threshold.

**Test cases:** (1) Normal: mid-range credit utilization. (2) Boundary: missing income imputed to median. (3) Adversarial: extreme outliers in debt-to-income after unit confusion (dollars vs cents).

**Measurement:** Recall on deny class, Brier score, and coefficient stability under bootstrap.

**Design question:** When would a tree ensemble beat linear regression without violating interpretability requirements?

**Predict before running.** The best model is the simplest one that meets the real decision requirement. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 2.2: Supervised Learning"""

X = [[1.0, 0.5], [1.0, 1.2], [1.0, 2.0]]
y = [0, 0, 1]
w = [0.0, 0.0]
lr = 0.3
for _ in range(30):
    for xi, yi in zip(X, y):
        z = sum(wj*xij for wj, xij in zip(w, xi))
        pred = 1 / (1 + pow(2.718281828, -z))
        err = pred - yi
        w = [wj - lr * err * xij for wj, xij in zip(w, xi)]
print("weights:", [round(v, 3) for v in w])


#### Interpret and apply

Implement linear and logistic regression before using a library.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why specify the loss function before choosing model architecture?
2. What symptom indicates regularization is too weak on small lending data?
3. What linear baseline anchors complex model comparisons?

<details><summary>Check your reasoning</summary>

Q1: Wrong loss optimizes accuracy while compliance needs recall on denies. Q2: Large train/val gap and wild coefficients. Q3: Logistic regression with L2 and same features.

</details>


**Section checkpoint.** Define a prediction cutoff and a naive baseline. Identify a feature available only after the cutoff.


## Representations and neural networks


### Lecture 03 · Unsupervised and Representation Learning

Use unlabeled data to discover structure, compress observations, and learn reusable representations.

#### A cluster is a clue

An unsupervised analysis finds a tight cluster of short requests submitted in the evening. The model labels it ‘low detail’. The tenant liaison suspects these are residents using an accessibility setting on the form.

Read sampled requests before turning cluster labels into operational categories. Compare source and task variables and check stability after removing time of day. A useful cluster can describe a form problem that no repair model should learn from.

#### Read the mechanism

**clustering**

Clustering groups unlabeled points by similarity—k-means, hierarchical, or density methods. Clusters are hypotheses about structure that require domain validation.

Grouping support tickets by embedding clusters reveals recurring themes but does not automatically name them correctly.

**dimensionality reduction**

Dimensionality reduction projects high-dimensional data to fewer dimensions for visualization, compression, or denoising—PCA, t-SNE, UMAP. Preserved geometry depends on the method.

PCA on ticket embeddings for dashboard visualization may linearly mix topics; UMAP preserves local neighborhoods differently.

**autoencoders**

Autoencoders learn compressed representations by reconstructing inputs through a bottleneck layer. They support anomaly detection and pretraining when labels are scarce.

Reconstruction error spikes on malformed log lines that never appeared in training—useful for anomaly alerts.

**self-supervision**

Self-supervision creates training signal from the data itself—mask prediction, contrastive pairs—without manual labels. It scales representation learning to massive unlabeled corpora.

BERT-style masked language modeling learns syntax and semantics from raw text before task fine-tuning.

**representation learning**

Representation learning discovers features automatically instead of hand-engineering them. Quality of representations determines retrieval, transfer, and sample efficiency.

Sentence embeddings trained on internal docs outperform bag-of-words on paraphrase-heavy policy search.

#### Worked example

**Situation:** Unlabeled merchant transaction narratives pile up; compliance wants emergent fraud motifs without predefined labels.

**Baseline:** Random cluster assignment—stable clusters but meaningless.

**Application:** Embed narratives with TF–IDF, k-means with k sweep, visualize with PCA, then manually validate whether clusters align with known fraud typologies or artifacts (merchant category codes).

**Test cases:** (1) Normal: clear separation of payroll vs retail vocabularies. (2) Boundary: k equals number of MCC codes—clusters mirror metadata not text. (3) Adversarial: duplicate boilerplate terms dominating centroids.

**Measurement:** Silhouette score, cluster purity vs small labeled audit set, and analyst time to narrate cluster meaning.

**Design question:** What evidence would convince you a cluster is a fraud hypothesis rather than a preprocessing artifact?

**Predict before running.** Structure found by an algorithm is a hypothesis to validate, not a fact. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 2.3: Unsupervised and Representation Learning"""

import math
docs = {"payroll": "salary direct deposit batch", "retail": "card swipe retail purchase"}
words = sorted(set(w for d in docs.values() for w in d.split()))
def vec(doc):
    counts = {w: doc.split().count(w) for w in words}
    df = {w: sum(w in d.split() for d in docs.values()) for w in words}
    n = len(docs)
    return {w: counts[w] * math.log(n / df[w]) for w in words}
va, vb = vec(docs["payroll"]), vec(docs["retail"])
dot = sum(va[w]*vb[w] for w in words)
print("payroll vs retail dot:", round(dot, 3))


#### Interpret and apply

Cluster a dataset, visualize it, and explain why clusters are not automatically meaningful categories.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why are unsupervised clusters hypotheses rather than ground truth?
2. How would you detect clusters driven by MCC metadata instead of text?
3. What random baseline shows structure is non-trivial?

<details><summary>Check your reasoning</summary>

Q1: No label alignment—clusters may reflect formatting artifacts. Q2: Cluster purity tracks MCC one-to-one. Q3: Random cluster labels with same k.

</details>


### Lecture 04 · Neural Networks

Build the mental model of layers, activations, losses, backpropagation, initialization, normalization, and optimization.

#### Watch the model learn who wrote the ticket

The neural model reads both description and channel. It quickly learns that phone reports are often urgent because a front-desk worker already screened them. A new contractor will begin transcribing all calls.

Retraining can reduce loss by learning an unstable shortcut. Compare a model with and without the channel feature and test it during a shift in intake practice. Preserve the option to use channel for monitoring even if the predictor should ignore it.

#### Read the mechanism

**neurons and layers**

Neurons apply activations to weighted sums; layers stack these transforms into composable functions. Depth lets networks build hierarchical abstractions.

First layers in vision nets detect edges; deeper layers combine them into parts and objects.

**activations**

Activation functions introduce nonlinearity—ReLU, GELU, sigmoid—without which deep networks collapse to linear maps. Choice affects gradient flow and training stability.

GELU in transformers smooths gradients compared to ReLU for language modeling at scale.

**backpropagation**

Backpropagation applies the chain rule to compute gradients through layered computations efficiently. It enables training deep networks but requires careful initialization and normalization.

One backward pass from loss to weights updates every layer in a classifier simultaneously.

**normalization**

Text normalization lowercases, strips diacritics, standardizes whitespace, and canonicalizes equivalents before indexing or tokenization. Over-normalization destroys discriminative identifiers.

Collapsing hyphens in SKUs merges distinct product codes; preserving case matters for camelCase APIs.

**optimizers**

Optimizers like Adam, AdamW, and SGD with momentum adapt update rules beyond vanilla gradient descent. They affect convergence speed, final loss, and generalization.

AdamW decouples weight decay from adaptive steps—common default for transformer fine-tuning.

#### Worked example

**Situation:** A neural scorer must capture nonlinear interactions among debt, income, and employment length for the lender's API.

**Baseline:** Single-layer logistic regression plateau on validation AUC.

**Application:** Train a two-hidden-layer MLP with ReLU, track train/val loss, inspect gradient norms for vanishing/exploding signals, apply batch normalization ablation.

**Test cases:** (1) Normal: batch size 64, stable learning rate. (2) Boundary: very small batch with noisy gradients. (3) Adversarial: all-zero input column after pipeline bug.

**Measurement:** Val AUC vs epoch, gradient norm percentiles, and latency per inference at batch 1.

**Design question:** At what point does adding layers stop improving the deny-recall slice?

**Predict before running.** Neural networks learn compositions of transformations; training adjusts those transformations to reduce loss. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 2.4: Neural Networks"""

def relu(x):
    return max(0.0, x)
x, w1, b1, w2, b2 = 1.5, 0.8, -0.2, 1.2, 0.1
hidden = relu(x * w1 + b1)
y = hidden * w2 + b2
loss = (y - 1.0) ** 2
grad_w2 = 2 * (y - 1.0) * hidden
print({"hidden": round(hidden, 3), "y": round(y, 3), "grad_w2": round(grad_w2, 3)})


#### Interpret and apply

Train a small network and inspect gradients and learning curves.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What does a flat validation curve alongside falling training loss suggest?
2. How would a zero-input-column bug manifest in gradients?
3. What non-neural baseline must the MLP beat?

<details><summary>Check your reasoning</summary>

Q1: Overfitting—capacity exceeds data signal. Q2: Weights on dead feature stay near init. Q3: Same-data logistic regression.

</details>


**Section checkpoint.** Compare fitting a line with learning an intermediate representation. Name an input for which a line would be inadequate.


## Evidence and lifecycle


### Lecture 05 · Evaluation and Error Analysis

Choose metrics from the decision context, estimate uncertainty, inspect slices, and turn mistakes into the next experiment.

#### Ninety per cent overall is a poor handoff

The pilot routes 90 of 100 tickets correctly. The mobile slice contributes 8 of the 10 failures even though it contributes only 20 tickets. Daniel cannot tell whether the issue is short text or an old adapter.

Report per-slice denominators and inspect the serialized request reaching the model. Fix a contract if the text was lost. If it arrived intact, compare error types and availability of supporting fields before adding model complexity.

#### Read the mechanism

**confusion matrix**

A confusion matrix counts predicted versus actual classes, exposing which errors dominate. It is essential when classes are imbalanced or costs asymmetric.

A router may confuse 'billing' with 'refund' while rarely missing 'outage'—the matrix shows where to invest labeling.

**precision and recall**

Precision is correctness among positive predictions; recall is coverage of actual positives. Trading them off reflects whether false positives or false negatives hurt more.

High recall in safety alerts catches more incidents; high precision in auto-replies avoids annoying customers.

**calibration**

Calibration means predicted probabilities align with observed frequencies—70% confidence should be right about 70% of the time. Uncalibrated scores mislead threshold and cost decisions.

A medical triage model with miscalibrated probabilities causes undertriage when 0.9 confidence actually means 0.6 accuracy.

**cross-validation**

Cross-validation rotates train and validation folds to estimate performance variance with limited data. It reduces luck from a single split but must respect temporal or group structure when required.

K-fold on i.i.d. tabular data estimates variance; time-series tasks need forward-chaining instead.

**slice analysis**

Slice analysis evaluates metrics on subpopulations—language, product, tenant—to catch aggregate illusions. A model can pass overall while failing high-value segments.

95% accuracy overall can hide 60% on enterprise accounts or non-English queries.

#### Worked example

**Situation:** Regulators ask why the model denies more applications in one region; aggregate AUC masks the disparity.

**Baseline:** Report global AUC only—hides regional recall collapse.

**Application:** Build confusion matrices overall and by region, compute recall@deny with Wilson confidence intervals, run slice analysis on income bands, write error taxonomy (data missing vs true risk vs score threshold).

**Test cases:** (1) Normal: balanced region with adequate sample size. (2) Boundary: region with n=30—wide confidence intervals. (3) Adversarial: proxy feature encoding zip code leading to disparate impact.

**Measurement:** Slice recall CIs, calibration by region, and taxonomy counts driving next experiment.

**Design question:** Which slice would you gate release on despite strong global AUC?

**Predict before running.** An aggregate metric can hide the exact population where a system is unsafe or useless. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 2.5: Evaluation and Error Analysis"""

confusion = {"TP": 40, "FP": 10, "FN": 8, "TN": 142}
def metrics(c):
    prec = c["TP"] / (c["TP"] + c["FP"] + 1e-9)
    rec = c["TP"] / (c["TP"] + c["FN"] + 1e-9)
    return round(prec, 3), round(rec, 3)
slice_b = {"TP": 5, "FP": 12, "FN": 6, "TN": 20}
print("overall:", metrics(confusion))
print("slice_b:", metrics(slice_b))


#### Interpret and apply

Write an error taxonomy and compare two models with confidence intervals.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why can aggregate AUC hide an unsafe slice?
2. What distinguishes calibration failure from threshold failure in one region?
3. What evaluation baseline uses only global accuracy?

<details><summary>Check your reasoning</summary>

Q1: Strong performance on majority slice dominates AUC. Q2: Calibration: miscalibrated scores; threshold: wrong operating point. Q3: Majority-class predictor on accuracy only.

</details>


### Lecture 06 · The ML Lifecycle

Connect data, experiments, models, releases, monitoring, drift, retraining, and retirement into an accountable lifecycle.

#### Assign the person who can roll back

A form release changes how the unit number is encoded. The model keeps scoring requests, but the technician tool cannot use half the routes.

Monitor the output contract and the completed handoff, not just model uptime. Tie training data, model, and form versions together. Give a named operator a reversible route to the queue staff already know.

#### Read the mechanism

**experiment tracking**

Experiment tracking logs hyperparameters, data versions, metrics, and artifacts for every training run. Without it, teams cannot reproduce or compare results.

Logging learning rate, seed, and dataset hash explains why run 47 beat run 46.

**model registry**

A model registry stores versioned models with stage labels—staging, production, archived—and metadata for audit. It is the handoff point between ML and serving teams.

Promoting v3.2 to production requires passing eval gates linked in the registry entry.

**data validation**

Data validation checks schema, ranges, distributions, and freshness of incoming data before training or inference. Silent schema drift breaks pipelines quietly.

A new optional field arriving as null for 40% of rows should block training until investigated.

**drift**

Drift is change in input or label distributions over time—covariate, prior, or concept drift. Unmonitored drift erodes model value without code changes.

New product vocabulary after a launch shifts ticket text while labels stay stable—covariate drift.

**monitoring**

Monitoring observes live inputs, outputs, latency, errors, and business metrics continuously. It connects production behavior to retraining and incident response.

A spike in abstention rate may signal upstream data breakage before users complain.

#### Worked example

**Situation:** The prediction service moves to production; six months later income verification vendor changes JSON schema silently.

**Baseline:** Deploy model once with no monitoring—silent feature nulling.

**Application:** Define ML lifecycle checklist: data validation on ingest, model registry version, shadow deploy, drift alarms on feature null rate and PSI, documented rollback to prior artifact.

**Test cases:** (1) Normal: weekly retrain with stable schema. (2) Boundary: partial null spike on one feature 2%→40%. (3) Adversarial: schema rename bypassing validation rules.

**Measurement:** Time-to-detect drift, rollback duration, and decision quality before/after rollback.

**Design question:** Which monitor fires first—data validation or outcome-based performance—and why?

**Predict before running.** A trained model is an artifact; value and risk emerge from its full operating system. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 2.6: The ML Lifecycle"""

CHAPTER = "2.6"
print("chapter hook:", CHAPTER)
registry = {"model_v3": {"features": ["income", "debt"]}}
live = {"income": None, "debt": 1200}
def validate(row, schema):
    return [f for f in schema["features"] if row.get(f) is None]
issues = validate(live, registry["model_v3"])
print({"issues": issues, "action": "rollback" if issues else "serve"})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Write a release checklist and a rollback plan for a prediction service.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What is the difference between a model artifact and its operating system?
2. How does schema drift appear before label drift?
3. What release process lacks rollback evidence?

<details><summary>Check your reasoning</summary>

Q1: Artifact is weights; operating system includes pipes, monitors, policy. Q2: Feature null rates spike while labels lag weeks. Q3: Direct promote without registry pin or canary.

</details>


**Section checkpoint.** Define a holdout slice and a rollback trigger. Explain which metric would detect the failure first.


## Course workshop

Add a held-out point and compare the trained line with the mean of the training labels. Keep the held-out point outside DATA.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Fit y = wx + b with batch gradient descent."""
DATA = [(0, 1), (1, 3), (2, 5), (3, 7)]
w = b = 0.0
rate = 0.05

for step in range(501):
    dw = db = loss = 0.0
    for x, y in DATA:
        error = (w * x + b) - y
        loss += error * error
        dw += 2 * error * x / len(DATA)
        db += 2 * error / len(DATA)
    w -= rate * dw
    b -= rate * db
    if step % 100 == 0:
        print(f"step={step:3d} loss={loss/len(DATA):.6f} w={w:.3f} b={b:.3f}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
held_out = (4, 9)
x, expected = held_out
prediction = w * x + b
mean_label = sum(y for _, y in DATA) / len(DATA)
assert abs(prediction - expected) < abs(mean_label - expected)
print('Workshop checks passed')


## Course project

Deliver a prediction service with a data card, baseline, evaluation report, error analysis, and monitoring plan.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
